# RAG Support Chatbot — Milestone 3 (VS Code / local version)
## Advanced Techniques & Deployment

This notebook **tests** the RAG chain interactively.
The actual production code lives in:
- `src/rag_chain.py` — core RAG logic (retrieval + generation)
- `src/api.py`       — FastAPI REST server wrapping the chain

**Pipeline per query:**
```
user question
  → embed with sentence-transformers (all-MiniLM-L6-v2)
  → hybrid retrieve top-3 from FAISS index
  → build prompt: system message + context docs + question
  → generate answer with flan-t5-base (local CPU)
  → return answer + sources
```

**Steps:**
```
Step 1 → Install new dependencies
Step 2 → Load all models (embedding + FAISS + LLM)
Step 3 → Test the RAG chain interactively
Step 4 → Evaluate answer quality
Step 5 → Test the REST API
Step 6 → Security notes
```

## Step 1 — Install new dependencies

In [1]:
# Run this once in your activated venv terminal:
#   pip install transformers torch fastapi uvicorn[standard] pydantic httpx
#
# torch is needed by transformers for flan-t5 inference on CPU.
# httpx is needed to test the API from inside the notebook.
#
# Uncomment to install from inside the notebook:
# %pip install transformers torch fastapi uvicorn[standard] pydantic httpx

In [2]:
print("Start")

Start


## Step 2 — Load all models

In [3]:
import importlib
import os
import sys

# Make sure Python can find the src/ package
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)


import src.rag_chain as rag_chain

# Reload the module to avoid using an older copy already cached in the notebook kernel.
rag_chain = importlib.reload(rag_chain)

load_all = rag_chain.load_all
ask = rag_chain.ask
search = rag_chain.search
rebuild_index = rag_chain.rebuild_index

# Load everything into memory.
# First run: downloads flan-t5-base (~250MB) from Hugging Face.
# Subsequent runs: loads from local cache — much faster.
load_all()
print('All models loaded and ready.')

c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\lojyn\OneDrive\Documents\GitHub\NHA-4-231\src\rag_chain.py:45: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


[RAG] Configuring Gemini API for embeddings...
[RAG] Gemini embedding model: gemini-embedding-001 (3072-dim)
[RAG] Initialising Groq client for generation...
[RAG] Groq ready: llama-3.3-70b-versatile
[RAG] Loading FAISS index...
[RAG] Index loaded: 800 vectors, dim=3072
[RAG] Loading train lookup table...
[RAG] Loading BM25 corpus...
[RAG] All components loaded. Ready.

All models loaded and ready.


In [ ]:
import nest_asyncio
nest_asyncio.apply()

# Run ONCE to rebuild the FAISS index from 384-dim to 768-dim Gemini embeddings.
# After it finishes, never need to run again unless you change the embedding model.
rebuild_index(batch_size=5, max_concurrent=1)

[RAG] Using 2 API key(s) for embedding,max_concurrent=1.
[RAG] Rebuilding FAISS index with gemini-embedding-001 (3072-dim)...
[RAG] 17,268 rows, batch_size=5
[RAG] Daily capacity: 1000 req/key x 5 rows = 5000 rows/key/day
[RAG] Checkpoint enabled — safe to re-run if interrupted.

[RAG] Resuming from row 5990 (5990 embeddings done)

[DEBUG] Total texts to embed: 17268
[DEBUG] Starting loop at index i = 5990


Embedding:   0%|          | 1/2256 [00:01<1:05:14,  1.74s/it]


[RAG] Key Index 0 -> Status Code: 200


Embedding:   0%|          | 2/2256 [00:02<40:22,  1.07s/it]  


[RAG] Key Index 0 -> Status Code: 200


...


[RAG] Key Index 0 -> Status Code: 429

[RAG] Key Index 0 rate limited (429). Switched pointer. Backing off for 60s (attempt 1/6)...

[RAG] Key Index 1 -> Status Code: 429

[RAG] Key Index 1 rate limited (429). Switched pointer. Backing off for 75s (attempt 2/6)...

[RAG] Key Index 0 -> Status Code: 429

[RAG] Key Index 0 rate limited (429). Switched pointer. Backing off for 90s (attempt 3/6)...

[RAG] Key Index 1 -> Status Code: 429

[RAG] Key Index 1 rate limited (429). Switched pointer. Backing off for 105s (attempt 4/6)...

[RAG] Key Index 0 -> Status Code: 429

[RAG] Key Index 0 rate limited (429). Switched pointer. Backing off for 120s (attempt 5/6)...

[RAG] Key Index 1 -> Status Code: 429

[RAG] Key Index 1 rate limited (429). Switched pointer. Backing off for 135s (attempt 6/6)...


Embedding:  18%|█▊        | 409/2256 [1:16:22<230:03:49, 448.42s/it]


[RAG] Key Index 0 -> Status Code: 429

[RAG] Key Index 0 rate limited (429). Switched pointer. Backing off for 60s (attempt 1/6)...

[RAG] Key Index 1 -> Status Code: 429

[RAG] Key Index 1 rate limited (429). Switched pointer. Backing off for 75s (attempt 2/6)...

[RAG] Key Index 0 -> Status Code: 429

[RAG] Key Index 0 rate limited (429). Switched pointer. Backing off for 90s (attempt 3/6)...

[RAG] Key Index 1 -> Status Code: 429

[RAG] Key Index 1 rate limited (429). Switched pointer. Backing off for 105s (attempt 4/6)...

[RAG] Key Index 0 -> Status Code: 429

[RAG] Key Index 0 rate limited (429). Switched pointer. Backing off for 120s (attempt 5/6)...

[RAG] Key Index 1 -> Status Code: 429

[RAG] Key Index 1 rate limited (429). Switched pointer. Backing off for 135s (attempt 6/6)...


Embedding:  18%|█▊        | 410/2256 [1:26:11<251:32:33, 490.55s/it]


[RAG] Key Index 0 -> Status Code: 429

[RAG] Key Index 0 rate limited (429). Switched pointer. Backing off for 60s (attempt 1/6)...

[RAG] Key Index 1 -> Status Code: 429

[RAG] Key Index 1 rate limited (429). Switched pointer. Backing off for 75s (attempt 2/6)...

[RAG] Key Index 0 -> Status Code: 429

[RAG] Key Index 0 rate limited (429). Switched pointer. Backing off for 90s (attempt 3/6)...


## Step 3 — Test the RAG chain interactively

In [13]:
def print_result(result: dict) -> None:
    """Pretty-print a RAG chain result dict."""
    print(f"QUERY    : {result['query']}")
    print(f"RETRIEVAL: {result['retrieval']}")
    print(f"\nANSWER:\n{result['answer']}")
    print(f"\nSOURCES USED ({len(result['sources'])} docs):")
    for i, src in enumerate(result['sources'], 1):
        print(f"  #{i} [{src['category']} -> {src['intent']}]  score={src['score']:.4f}")
        print(f"      Q: {src['instruction']}")
        print(f"      A: {src['response'][:100]}...")
    print('='*65)

In [14]:
def ask_bm25_only(query: str, top_k: int = 3) -> dict:
    """
    Temporary fallback: BM25 retrieval + Gemini generation.
    No embedding API calls — works even when quota is exhausted.
    """
    import re
    from rank_bm25 import BM25Okapi
    
    tokens = re.findall(r'\b\w+\b', query.lower())
    scores = rag_chain._bm25.get_scores(tokens)
    top_indices = scores.argsort()[::-1][:top_k]
    
    context_docs = []
    for idx in top_indices:
        row = rag_chain._train_df.iloc[idx]
        context_docs.append({
            "score"      : float(scores[idx]),
            "category"   : row["category"],
            "intent"     : row["intent"],
            "instruction": row["instruction_clean"],
            "response"   : row["response_clean"],
        })
    
    prompt = rag_chain._build_prompt(query, context_docs)
    answer = rag_chain._generate(prompt)
    
    return {
        "query"    : query,
        "answer"   : answer,
        "sources"  : context_docs,
        "retrieval": "bm25_only",
    }

In [15]:
result = ask_bm25_only("I want to cancel my order")
print_result(result)

QUERY    : I want to cancel my order
RETRIEVAL: bm25_only

ANSWER:
I'd be happy to help you with canceling your order. To proceed, could you please provide me with your order number? Additionally, I'll need you to follow these steps:

1. Log in to your account using your credentials.
2. Navigate to the "Order History" or "My Orders" section.
3. Locate the order you wish to cancel and click on it.
4. Look for the option to "Cancel Order" and select it.
5. If prompted, provide any required information or reason for cancellation.
6. Confirm the cancellation to complete the process.

If you encounter any difficulties or have further questions, our dedicated support team is available to assist you. You can reach us at our support line or through the Live Chat feature on our official website. We're committed to ensuring your satisfaction and will do our best to help you throughout the cancellation process.

SOURCES USED (3 docs):
  #1 [ORDER -> cancel_order]  score=14.9065
      Q: assistanc

In [16]:
# --- Test 1: Order cancellation ---
result = ask("I want to cancel my order")
print_result(result)

[RAG] Embedding quota exhausted — using BM25 fallback for retrieval.
QUERY    : I want to cancel my order
RETRIEVAL: hybrid

ANSWER:
I'd be happy to help you with canceling your order. To proceed, could you please provide me with your order number? Additionally, I'll guide you through the steps to cancel your order. 

Please follow these steps:
1. Log in to your account using your credentials.
2. Navigate to the "Order History" or "My Orders" section.
3. Locate the order you wish to cancel and click on it.
4. Look for the option to "Cancel Order" and select it.
5. If prompted, provide any required information or reason for cancellation.
6. Finally, confirm the cancellation to complete the process.

If you encounter any difficulties or have further questions, our dedicated support team is available to assist you. You can reach us at our support line or through the Live Chat feature on our official website. We're committed to ensuring your satisfaction and will do our best to help you th

In [17]:
# --- Test 2: Missing package ---
result = ask("My package hasn't arrived and it's been 2 weeks")
print_result(result)

[RAG] Embedding quota exhausted — using BM25 fallback for retrieval.
QUERY    : My package hasn't arrived and it's been 2 weeks
RETRIEVAL: hybrid

ANSWER:
I'm so sorry to hear that your package hasn't arrived yet. According to our refund policy, if your order hasn't arrived within the expected timeframe, you may be eligible for a refund. Since it's been 2 weeks, I recommend reaching out to our customer support team directly so we can investigate the matter further and determine the best course of action. Please have your order number ready, and we'll do our best to assist you with a refund or provide an update on the status of your package. Would you like me to provide you with the contact information for our customer support team?

SOURCES USED (3 docs):
  #1 [REFUND -> check_refund_policy]  score=19.0235
      Q: see in what situations can I request my money back
      A: Of course! Let's explore the situations in which you can request a refund. Rest assured, we prioriti...
  #2 [REF

In [18]:
# --- Test 3: Password reset ---
result = ask("I forgot my password and I cannot log into my account")
print_result(result)

[RAG] Embedding quota exhausted — using BM25 fallback for retrieval.
QUERY    : I forgot my password and I cannot log into my account
RETRIEVAL: hybrid

ANSWER:
I'm here to help you recover your account password. To get started, please navigate to the login page on our platform and locate the "Forgot Password" option. Click on it, and you will be guided through the necessary steps to reset your password. You will be prompted to enter the email address associated with your account, and then you will receive an email with instructions on how to reset your password. If you don't see the email in your inbox, please check your spam or junk folder. If you encounter any issues or have any questions during the process, don't hesitate to reach out to me. I'm here to assist you every step of the way and ensure a smooth password recovery experience for you.

SOURCES USED (3 docs):
  #1 [ACCOUNT -> recover_password]  score=23.9229
      Q: i try to recover my account pass
      A: I realized, you 

In [19]:
# --- Test 4: Double charge ---
result = ask("I was charged twice for the same order")
print_result(result)

[RAG] Embedding quota exhausted — using BM25 fallback for retrieval.
QUERY    : I was charged twice for the same order
RETRIEVAL: hybrid

ANSWER:
I'm so sorry to hear that you were charged twice for the same order. I'm here to help you with that. According to our refund policy, if you were charged multiple times for the same product or service, you can request a refund for the duplicate charges. To assist you further, could you please provide me with your order number and any other relevant details about the transaction? This will help me to look into this matter and guide you through the process of requesting a refund for the duplicate charge.

SOURCES USED (3 docs):
  #1 [REFUND -> check_refund_policy]  score=21.0741
      Q: I'm trying to see in which cases can i request refunds
      A: I truly appreciate your diligence in understanding the refund process. Let me assist you in clarifyi...
  #2 [ORDER -> track_order]  score=17.3111
      Q: see eta of order your order number
      A

In [20]:
# --- Test 5: Defective product return ---
result = ask("The product I received is broken, how do I return it?")
print_result(result)

[RAG] Embedding quota exhausted — using BM25 fallback for retrieval.
QUERY    : The product I received is broken, how do I return it?
RETRIEVAL: hybrid

ANSWER:
I'm so sorry to hear that the product you received is broken. I'm here to help you with the return process. Since the product is defective, you may be entitled to a refund. To initiate the return process, could you please provide me with your order number and a brief description of the issue with the product? This will allow me to guide you through the next steps and ensure that you receive a refund or a replacement as soon as possible. Additionally, I'll need to know if you have the original packaging and any accessories that came with the product, as this may be required for the return. Let's work together to resolve this issue and get you a working product or a refund, whichever you prefer.

SOURCES USED (3 docs):
  #1 [REFUND -> get_refund]  score=24.3201
      Q: I am trying to obtain a reimbursement
      A: I'm clearly c

In [21]:
# --- Test 6: Noisy/informal query (simulates real customer typing) ---
result = ask("whr is my ordr?? its been forever")
print_result(result)

[RAG] Embedding quota exhausted — using BM25 fallback for retrieval.
QUERY    : whr is my ordr?? its been forever
RETRIEVAL: hybrid

ANSWER:
I completely understand your concern about the status of your order, and I'm here to help. To provide you with the most accurate information, could you please provide me with your order number or any other relevant details about your order? This will allow me to check on the status and give you a precise update on when you can expect your package to arrive. Your patience is greatly appreciated, and I'm looking forward to assisting you further. How has your experience been with our service so far?

SOURCES USED (3 docs):
  #1 [DELIVERY -> delivery_period]  score=9.0751
      Q: can i check when my package is gonna arrive
      A: We completely understand your eagerness to track the arrival of your package and determine its estim...
  #2 [DELIVERY -> delivery_period]  score=8.6792
      Q: I need help seeing how long it takes for my item to arrive
 

## Step 4 — Evaluate answer quality

We compare the LLM-generated answer against the gold response from the dataset.
A high ROUGE score means the generated answer closely mirrors the expected response.

In [22]:
import pandas as pd
import numpy as np
import nltk
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

rouge  = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

test_df = pd.read_csv('../data/test_df.csv')

EVAL_SAMPLE = 50   # keep small — each row calls the LLM which is slow on CPU
sample = test_df.sample(EVAL_SAMPLE, random_state=42).reset_index(drop=True)

bleu_scores, r1, r2, rl = [], [], [], []

print(f'Evaluating {EVAL_SAMPLE} test queries end-to-end (retrieve + generate)...')
print('This takes a few minutes on CPU — each query runs the full LLM pipeline.\n')

for _, row in tqdm(sample.iterrows(), total=EVAL_SAMPLE):
    result      = ask(row['instruction_clean'], top_k=3)
    gold        = str(row['response_clean'])
    generated   = result['answer']

    # BLEU
    ref = nltk.word_tokenize(gold.lower())
    hyp = nltk.word_tokenize(generated.lower())
    bleu_scores.append(sentence_bleu([ref], hyp, smoothing_function=smooth))

    # ROUGE
    rs = rouge.score(gold, generated)
    r1.append(rs['rouge1'].fmeasure)
    r2.append(rs['rouge2'].fmeasure)
    rl.append(rs['rougeL'].fmeasure)

print('\n' + '='*50)
print('END-TO-END EVALUATION RESULTS (RAG chain)')
print('='*50)
print(f'  Sample size : {EVAL_SAMPLE}')
print(f'  Avg BLEU    : {np.mean(bleu_scores):.4f}')
print(f'  Avg ROUGE-1 : {np.mean(r1):.4f}')
print(f'  Avg ROUGE-2 : {np.mean(r2):.4f}')
print(f'  Avg ROUGE-L : {np.mean(rl):.4f}')
print()
print('Score guide for this task:')
print('  ROUGE-1 > 0.40 = good retrieval   ROUGE-L > 0.35 = coherent generation')

ModuleNotFoundError: No module named 'rouge_score'

## Step 5 — Test the REST API

**Before running this step**, start the API server in a separate VS Code terminal:

```bash
# Make sure venv is active, then from the project root:
uvicorn src.api:app --reload --port 8000
```

Wait until you see:
```
[API] Ready to serve requests.
INFO:     Application startup complete.
```

Then run the cells below.

In [ ]:
import httpx
import json

BASE_URL = "http://localhost:8000"

# --- Health check ---
resp = httpx.get(f"{BASE_URL}/health")
print("GET /health")
print(json.dumps(resp.json(), indent=2))

In [ ]:
# resp = httpx.get(f"{BASE_URL}/health")
# print(f"Status code: {resp.status_code}")
# print(f"Raw response: {resp.text}")

In [ ]:
# --- POST /ask ---
payload = {
    "question"  : "I want to cancel my order",
    "top_k"     : 3,
    "use_hybrid": True
}
resp = httpx.post(f"{BASE_URL}/ask", json=payload, timeout=60)
data = resp.json()

print("POST /ask")
print(f"  Query  : {data['query']}")
print(f"  Answer : {data['answer']}")
print(f"  Sources: {len(data['sources'])} docs retrieved")
for src in data['sources']:
    print(f"    [{src['intent']}]  score={src['score']:.4f}")

In [ ]:
# --- GET /search (retrieval only, no generation) ---
resp = httpx.get(f"{BASE_URL}/search", params={"query": "track my delivery", "top_k": 3})
print("GET /search?query=track my delivery&top_k=3")
for doc in resp.json():
    print(f"  [{doc['intent']}]  score={doc['score']:.4f}  ->  {doc['instruction']}")

In [ ]:
# --- Swagger UI shortcut ---
# You can also test the API interactively in your browser at:
print("Interactive API docs (Swagger UI):")
print(f"  {BASE_URL}/docs")
print()
print("Raw OpenAPI schema:")
print(f"  {BASE_URL}/openapi.json")

## Step 6 — Security notes

Your Milestone 3 requirements include: *'Secure endpoints with Azure AD or API keys'*.
Since we're running locally (no Azure yet), here is how security is handled:

**Current state (local dev):**
- CORS is open (`allow_origins=["*"]`) — fine locally, must be restricted before production
- No authentication on endpoints — intentional for local testing

**When Azure is fixed — production security checklist:**

| Layer | Local (now) | Azure (later) |
|---|---|---|
| Auth | None | Azure AD OAuth2 / API Management keys |
| CORS | `*` | Lock to your support portal domain |
| Transport | HTTP | HTTPS via Azure App Service |
| Rate limiting | None | Azure API Management policies |
| Secrets | None needed | Azure Key Vault |

**Quick local API key (optional, to show in the project):**
You can add a simple API key check to `src/api.py` by adding
this header dependency to each endpoint:
```python
from fastapi.security.api_key import APIKeyHeader
api_key_header = APIKeyHeader(name="X-API-Key")

async def verify_key(key: str = Depends(api_key_header)):
    if key != os.environ.get("API_KEY", "dev-key"):
        raise HTTPException(status_code=403, detail="Invalid API key")
```
Then set `API_KEY=your-secret` as an environment variable before running uvicorn.

## Milestone 3 — Summary

| Deliverable | Status | Where |
|---|---|---|
| RAG chain (retrieve + generate) | Done | `src/rag_chain.py` |
| REST API (`/ask`, `/search`, `/health`) | Done | `src/api.py` |
| Interactive testing | Done | this notebook |
| End-to-end evaluation (BLEU, ROUGE) | Done | Step 4 |
| Security plan | Done | Step 6 |
| Azure deployment | Pending (Azure issue) | `src/api.py` is Azure App Service ready |

**Project folder structure so far:**
```
NHA-4-231/
├── src/
│   ├── __init__.py
│   ├── rag_chain.py         <- core RAG logic
│   └── api.py               <- FastAPI REST server
├── notebooks/
│   ├── Milestone_1_VSCode.ipynb
│   ├── Milestone_2_Local.ipynb
│   └── Milestone_3_Local.ipynb  <- this file
├── data/
│   ├── train_df.csv / val_df.csv / test_df.csv
│   └── faiss_index/         <- FAISS index + embeddings from M2
└── venv/
```

---
**Next -> Milestone 4:** MLflow experiment tracking, monitoring dashboard,
and automated retraining pipeline.